# Support Ticket Agent
This notebook demonstrates the flow of utlizing multiple agents to


* Summarize top user issue
* Identify the root causes
* Provide top recommendations



## Setup

In [6]:
pip install langdetect

In [8]:
import os
import kagglehub
import pandas as pd
import random
import asyncio
import re
import json
import google.generativeai as genai
from google.colab import userdata

# API setup

api_key = userdata.get('API_Agent')
genai.configure(api_key=api_key)

model = genai.GenerativeModel("gemini-2.5-flash-lite")

## 0. Download Dataset- Kaggle Customer Support on Twitter

In [9]:
path = kagglehub.dataset_download("thoughtvector/customer-support-on-twitter")
print("Path to dataset files:", path)

Using Colab cache for faster access to the 'customer-support-on-twitter' dataset.
Path to dataset files: /kaggle/input/customer-support-on-twitter


In [10]:
file_path = os.path.join(path + '/twcs', "twcs.csv")
df = pd.read_csv(file_path)
print(df.head())
print(df.shape)

   tweet_id   author_id  inbound                      created_at  \
0         1  sprintcare    False  Tue Oct 31 22:10:47 +0000 2017   
1         2      115712     True  Tue Oct 31 22:11:45 +0000 2017   
2         3      115712     True  Tue Oct 31 22:08:27 +0000 2017   
3         4  sprintcare    False  Tue Oct 31 21:54:49 +0000 2017   
4         5      115712     True  Tue Oct 31 21:49:35 +0000 2017   

                                                text response_tweet_id  \
0  @115712 I understand. I would like to assist y...                 2   
1      @sprintcare and how do you propose we do that               NaN   
2  @sprintcare I have sent several private messag...                 1   
3  @115712 Please send us a Private Message so th...                 3   
4                                 @sprintcare I did.                 4   

   in_response_to_tweet_id  
0                      3.0  
1                      1.0  
2                      4.0  
3                      5.0  
4

## Phase 1: Data Preparation

In [11]:
# Extract customer initial complaints
df_user_init_complaint = (
    df[(df.inbound == True) & df.in_response_to_tweet_id.isnull() & df.response_tweet_id.notnull()]
)

df_user_init_complaint["company_reply_tweet_id"] = (
    df_user_init_complaint["response_tweet_id"]
    .astype(str)
    .str.split(",")
    .str[0]
)

# Join to company
df_company = (
    df[df.inbound == False][['tweet_id', 'author_id']]
    .rename(columns={'tweet_id': 'company_reply_tweet_id', 'author_id': 'company'})
)
df_company['company_reply_tweet_id'] = df_company['company_reply_tweet_id'].astype(str)

df_user_init_complaint = pd.merge(df_user_init_complaint, df_company, on='company_reply_tweet_id', how='left')

#Select sample company- Airbnb
df_user_init_complaint = df_user_init_complaint[df_user_init_complaint.company == 'AirbnbHelp']

print(f"Total AirbnbHelp complaints: {len(df_user_init_complaint)}")

/tmp/ipykernel_4671/1844617743.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_user_init_complaint["company_reply_tweet_id"] = (


Total AirbnbHelp complaints: 5577


### Data cleaning

In [19]:
def clean_tweet(text):
    text = str(text)
    text = re.sub(r'@\w+', '', text)           # remove @mentions
    text = re.sub(r'http\S+|www\S+', '', text)  # remove URLs
    text = re.sub(r'&amp;', '', text)            # remove &amp;
    text = re.sub(r'&\w+;', '', text)            # remove other HTML entities
    text = text.encode("ascii", "ignore").decode()  # remove non-ASCII
    text = text.lower()
    text = re.sub(r'!+', '!', text)
    text = re.sub(r'\?+', '?', text)
    text = re.sub(r'\.+', '.', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

df_main = df_user_init_complaint.copy()
df_main = df_main.dropna(subset=['text'])
df_main['clean_text'] = df_main['text'].astype(str).apply(clean_tweet)
df_main = df_main[df_main.clean_text.str.len() >= 20]

#Select 100 samples for demonstration
df_sample = df_main.sample(100, random_state=42)

In [20]:
#Filter out to only English user comments
from langdetect import detect
from langdetect.lang_detect_exception import LangDetectException

def is_english(text):
    try:
        return detect(str(text)) == "en"
    except LangDetectException:
        return False

df_sample = df_sample[df_sample["clean_text"].apply(is_english)].copy()
print(f"English complaints: {len(df_sample)}")

English complaints: 95


## Phase 2: Clustering
Instead of asking LLM to summarize issues into clusters, which consumes lots of tokens, I firstly
- Convert sentences into vectors using SentenceTransformer
- Then cluster the complaints into different clusters using BERTopics clustering technique.

In [17]:
#pip install bertopic

In [27]:
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
texts = df_sample["clean_text"].tolist()
embeddings = embedding_model.encode(texts, show_progress_bar=True)
print("Embedding shape:", embeddings.shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Embedding shape: (95, 384)


In [28]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction import text as sklearn_text

#Self defined stop words
custom_stop_words = [
    "airbnb", "help", "please", "thank", "hi", "hey",
    "just", "really", "would", "could", "get", "got"
]

#Include self-defined stop words
all_stop_words = sklearn_text.ENGLISH_STOP_WORDS.union(set(custom_stop_words))

vectorizer_model = CountVectorizer(
    stop_words  = list(all_stop_words),      # English stop words
    ngram_range = (1, 3),
    min_df      = 2               # only count if >=2 appearances
)

# BERTopic automatically decide the optimal clusters
topic_model = BERTopic(
    embedding_model    = embedding_model,
    vectorizer_model = vectorizer_model,
    language           = "english",
    calculate_probabilities = False,
    min_topic_size          = 3,
    verbose            = True
)

topics, probs = topic_model.fit_transform(texts, embeddings)
df_sample["cluster_id"] = topics

print(f"Topics found: {topic_model.get_topic_info().shape[0] - 1}")  # -1 excludes outlier topic
print(topic_model.get_topic_info().head(10))

2026-05-12 02:45:31,932 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-05-12 02:45:32,075 - BERTopic - Dimensionality - Completed ✓
2026-05-12 02:45:32,076 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-05-12 02:45:32,086 - BERTopic - Cluster - Completed ✓
2026-05-12 02:45:32,091 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-05-12 02:45:32,109 - BERTopic - Representation - Completed ✓


Topics found: 7
   Topic  Count                              Name  \
0     -1     15          -1_stay_price_host_guest   
1      0     32    0_host_refund_service_customer   
2      1     16  1_reservation_guest_booking_book   
3      2      9          2_feel_good_terrible_bad   
4      3      7            3_ive_dm_reply_account   
5      4      7         4_place_pay_apartment_fee   
6      5      6      5_listing_list_request_hosts   
7      6      3     6_didn_time_extenuating_asked   

                                      Representation  \
0  [stay, price, host, guest, offer, tomorrow, de...   
1  [host, refund, service, customer, problem, acc...   
2  [reservation, guest, booking, book, policy, do...   
3  [feel, good, terrible, bad, previous, disappoi...   
4  [ive, dm, reply, account, way, number, asap, l...   
5  [place, pay, apartment, fee, booked, need, hot...   
6  [listing, list, request, hosts, assist, theres...   
7  [didn, time, extenuating, asked, making, lost,...   

 

### Identify top keywords from each cluster

In [29]:
# Use keywords extraction from BERTopic
topic_info = topic_model.get_topic_info()

cluster_keywords_result = []
for _, row in topic_info.iterrows():
    topic_id = row["Topic"]
    if topic_id == -1:
        continue   # -1 是 outlier cluster from BERTopic

    # Extract keywords from BERTopic
    keywords = [word for word, _ in topic_model.get_topic(topic_id)][:6]
    cluster_texts = df_sample[df_sample.cluster_id == topic_id]["clean_text"]

    cluster_keywords_result.append({
        "cluster_id"   : topic_id,
        "keyword_label": " / ".join(keywords[:3]),
        "top_keywords" : keywords,
        "cluster_size" : len(cluster_texts)
    })

cluster_keywords_df = pd.DataFrame(cluster_keywords_result)

df_sample = pd.merge(
    df_sample,
    cluster_keywords_df[["cluster_id", "keyword_label", "top_keywords", "cluster_size"]],
    on="cluster_id", how="left"
)

print(cluster_keywords_df)

   cluster_id                  keyword_label  \
0           0        host / refund / service   
1           1  reservation / guest / booking   
2           2         feel / good / terrible   
3           3               ive / dm / reply   
4           4        place / pay / apartment   
5           5       listing / list / request   
6           6      didn / time / extenuating   

                                        top_keywords  cluster_size  
0  [host, refund, service, customer, problem, acc...            32  
1  [reservation, guest, booking, book, policy, does]            16  
2  [feel, good, terrible, bad, previous, disappoi...             9  
3             [ive, dm, reply, account, way, number]             7  
4         [place, pay, apartment, fee, booked, need]             7  
5    [listing, list, request, hosts, assist, theres]             6  
6     [didn, time, extenuating, asked, making, lost]             3  


In [30]:
df_sample.groupby('cluster_id').size()

,0
cluster_id,
-1,15
0,32
1,16
2,9
3,7
4,7
5,6
6,3


In [31]:
#Remove data from cluster= -1 (noise clusters)
df_sample = df_sample[df_sample['cluster_id']!=-1]
cluster_keywords_df = cluster_keywords_df[cluster_keywords_df['cluster_id']!=-1]

## Phase 3: Async Map-Reduce Agent Pipeline
This code is used to minimize the tokens burnt by using parallelized structure.

**Architecture:**
```
All clusters
     │
     ├── Chunk 1 (3 clusters) ──→ MAP Agent ──┐
     ├── Chunk 2 (3 clusters) ──→ MAP Agent ──┼──→ REDUCE Agent ──→ Final Report
     └── Chunk N (3 clusters) ──→ MAP Agent ──┘
            ↑ staggered + semaphore controlled
```
- Each MAP agent sees only a small chunk → no token limit
- All MAP calls fire simultaneously → faster
- REDUCE agent sees only compressed summaries → tiny payload

In [34]:
# Retrieve cluster examples

def get_cluster_examples(df, cluster_id, top_n=3):
    """Return top_n representative complaints, truncated to 150 chars each."""
    return (
        df[df.cluster_id == cluster_id]['clean_text']
        .astype(str)
        .head(top_n)
        .apply(lambda t: t[:150])
        .tolist()
    )


In [35]:
# ── Reusable Map-Reduce Engine ────────────────────────────────────────────────
# Credit: Claude

async def map_reduce(
    items           : list[dict],   # any data
    map_prompt_fn,                  # function(chunk) → prompt string
    reduce_prompt_fn,               # function(summaries) → prompt string
    chunk_size      : int = 3,
    concurrent      : int = 1,
    chunk_gap       : int = 8,
    max_retries     : int = 3,
) -> tuple[list[dict], dict]:
    """
    Generic Map-Reduce engine for any LLM analysis task.

    Usage:
        map_results, final = await map_reduce(
            items            = my_data,
            map_prompt_fn    = lambda chunk: f"Analyze this: {chunk}",
            reduce_prompt_fn = lambda summaries: f"Synthesize: {summaries}",
        )
    """

    # ── MAP ───────────────────────────────────────────────────────────────────
    chunks    = [items[i:i+chunk_size] for i in range(0, len(items), chunk_size)]
    semaphore = asyncio.Semaphore(concurrent)
    print(f"MAP: {len(items)} items → {len(chunks)} chunks of ~{chunk_size}")

    async def _call_with_retry(prompt, label):
        for attempt in range(max_retries):
            try:
                response = await asyncio.to_thread(model.generate_content, prompt)

                # 新增：檢查 response 是否為空
                if not response.text or not response.text.strip():
                    raise ValueError("Empty response from model")

                raw = response.text.replace("```json","").replace("```","").strip()
                return json.loads(raw)
            except Exception as e:
                wait = (2 ** attempt) + random.uniform(0, 1)
                print(f"  [{label}] Attempt {attempt+1} failed: {e}. Retrying in {wait:.1f}s...")
                await asyncio.sleep(wait)
        print(f"  [{label}] All retries failed")
        return {}

    async def _map_chunk(chunk, idx):
        await asyncio.sleep(idx * chunk_gap)   # stagger to avoid quota burst
        async with semaphore:
            prompt = map_prompt_fn(chunk)
            result = await _call_with_retry(prompt, f"Chunk {idx}")
            return result if isinstance(result, list) else [result]

    chunk_results = await asyncio.gather(*[_map_chunk(c, i) for i, c in enumerate(chunks)])
    map_results   = [item for sublist in chunk_results for item in sublist]
    print(f"MAP complete: {len(map_results)} results")

    # ── REDUCE ────────────────────────────────────────────────────────────────
    print(f"REDUCE: synthesizing {len(map_results)} results...")
    reduce_prompt = reduce_prompt_fn(map_results)
    final_result  = await _call_with_retry(reduce_prompt, "Reduce")
    print("REDUCE complete ✓")

    return map_results, final_result


In [36]:
# ── Support Ticket Pipeline ───────────────────────────────────────────────────

async def run_pipeline(df, cluster_keywords_df):
    """Run the full Map-Reduce agent pipeline for support ticket analysis."""

    # Prepare items — compact payload per cluster
    items = [
        {
            "cluster_id"  : int(row["cluster_id"]),
            "cluster_size": row["cluster_size"],
            "keywords"    : row["top_keywords"][:6],
            "examples"    : get_cluster_examples(df, int(row["cluster_id"]), top_n=3)
        }
        for _, row in cluster_keywords_df.iterrows()
    ]

    # MAP prompt — what each chunk agent sees
    def map_prompt(chunk):
        return f"""
        You are a customer support analyst. Analyze these complaint clusters.

        Clusters:
        {json.dumps(chunk, ensure_ascii=False, indent=2)}

        Return ONLY a JSON array — no explanation, no markdown fences:
        [
          {{
            "cluster_id": 0,
            "issue_label": "short label (3-5 words)",
            "issue_description": "one sentence",
            "likely_root_cause": "one sentence",
            "business_severity": "Low | Medium | High",
            "recommended_action": "one sentence",
            "customer_pain_point": "one sentence"
          }}
        ]
        """

    # REDUCE prompt — what the final synthesis agent sees
    def reduce_prompt(summaries):
        return f"""
        You are a senior customer support analyst writing an executive report.

        Below are analyses of {len(summaries)} customer complaint clusters.
        Synthesize them into a concise report.

        Cluster analyses:
        {json.dumps(summaries, ensure_ascii=False, indent=2)}

        Return ONLY a JSON object — no explanation, no markdown fences:
        {{
          "top_issues": [
            {{"rank": 1, "issue": "...", "affected_clusters": [...], "severity": "High | Medium | Low"}}
          ],
          "root_causes": [
            {{"rank": 1, "cause": "...", "related_issues": [...]}}
          ],
          "recommendations": [
            {{"priority": 1, "action": "...", "expected_impact": "...", "effort": "Low | Medium | High"}}
          ],
          "executive_summary": "2-3 sentence overall summary"
        }}
        """

    return await map_reduce(
        items            = items,
        map_prompt_fn    = map_prompt,
        reduce_prompt_fn = reduce_prompt,
        chunk_size       = 3,
        concurrent       = 1,
        chunk_gap        = 5,
    )


# Run pipeline code
map_summaries, final_report = await run_pipeline(df_sample, cluster_keywords_df)


MAP: 7 items → 3 chunks of ~3
MAP complete: 7 results
REDUCE: synthesizing 7 results...
REDUCE complete ✓


## Results

In [38]:
# Per-cluster results
cluster_agent_df = pd.DataFrame(map_summaries)
print("=== CLUSTER-LEVEL RESULTS ===")
print(cluster_agent_df[['cluster_id', 'issue_label', 'business_severity', 'likely_root_cause']].to_string(index=False))

=== CLUSTER-LEVEL RESULTS ===
 cluster_id                                  issue_label business_severity                                                                                                                                   likely_root_cause
          0              Host Service and Account Issues              High     Inconsistent host behavior and potential gaps in customer support processes for managing host-initiated cancellations and refund discrepancies.
          1             Reservation and Booking Disputes            Medium                Potential miscommunication or lack of clarity regarding booking policies, guest responsibilities, and dispute resolution procedures.
          2   Dissatisfaction with Accommodation Quality              High                                  Inaccurate or misleading listing descriptions, leading to unmet customer expectations and a poor guest experience.
          3                Direct Message Support Issues              High    

In [39]:
# Final report
print("=== EXECUTIVE SUMMARY ===")
print(final_report.get('executive_summary', 'N/A'))

print("\n=== TOP ISSUES ===")
for issue in final_report.get('top_issues', []):
    print(f"  #{issue['rank']} [{issue['severity']}] {issue['issue']}")

print("\n=== ROOT CAUSES ===")
for cause in final_report.get('root_causes', []):
    print(f"  #{cause['rank']} {cause['cause']}")

print("\n=== RECOMMENDATIONS ===")
for rec in final_report.get('recommendations', []):
    print(f"  P{rec['priority']} [{rec['effort']} effort] {rec['action']}")
    print(f"      Impact: {rec['expected_impact']}")

=== EXECUTIVE SUMMARY ===
Customer feedback highlights significant concerns in host reliability, accommodation quality accuracy, and pricing transparency, all contributing to high severity complaints. Addressing these core issues, alongside improving direct message support and host tools, is crucial for enhancing customer satisfaction and reducing financial disputes.

=== TOP ISSUES ===
  #1 [High] Host Service and Account Issues, including cancellations and refund requests
  #2 [High] Dissatisfaction with Accommodation Quality due to listing discrepancies
  #3 [High] Unexpected Additional Fees not clearly disclosed upfront
  #4 [High] Direct Message Support Issues, leading to delays and lack of response
  #5 [Medium] Reservation and Booking Disputes due to unclear policies
  #6 [Medium] Limitations in Host Tools, impacting listing management

=== ROOT CAUSES ===
  #1 Inconsistent host behavior and gaps in support processes for cancellations and refunds
  #2 Inaccurate or misleading li

In [40]:
# Save outputs
from google.colab import drive
drive.mount('/content/drive')

import os
save_dir = '/content/drive/MyDrive/Gen-AI/support-agent'
os.makedirs(save_dir, exist_ok=True)

Mounted at /content/drive


In [41]:
df_sample.to_csv(f'{save_dir}/df_sample.csv', index=False)

cluster_agent_df.to_csv(f'{save_dir}/cluster_analysis.csv', index=False)

with open(f'{save_dir}/final_report.json', 'w') as f:
    json.dump(final_report, f, indent=2)

print(f'Saved to {save_dir}')

Saved to /content/drive/MyDrive/Gen-AI/support-agent
